# San Antonio bicyclist crash hotspot areas

This notebook creates a simple, reproducible hotspot map for qualifying bicyclist crashes from 2021 through 2026. It groups crashes that occur within 500 feet of one another. That is a transparent recreation of the City's concentration-based Vision Zero work—not a claim that this is the City's exact unpublished formula.

Deaths and suspected serious injuries are counted separately. 2026 is partial-year coverage.


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


In [ ]:
# Load CRIS and keep pedalcyclists killed or suspected seriously injured.
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['City'] == 'SAN ANTONIO') &
             (raw['Person Type'] == '3 - PEDALCYCLIST') &
             (raw['Person Injury Severity'].isin(severity))].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
people = target[target['year'].between(2021, 2026)].copy()
crashes = (people.groupby('Crash ID', as_index=False)
           .agg(year=('year','first'), latitude=('latitude','first'), longitude=('longitude','first'),
                deaths=('death','sum'), serious_injuries=('serious_injury','sum'))
           .dropna(subset=['latitude','longitude']))
print('Qualifying crashes:', crashes['Crash ID'].nunique())
print('Affected bicyclists:', crashes['deaths'].sum() + crashes['serious_injuries'].sum())


## Create grouped hotspot areas

The grouping rule is intentionally easy to explain: crashes within 500 feet of each other are treated as one candidate area, and an area must contain at least two crashes. This identifies concentrations; it does not prove that every crash had the same cause.


In [ ]:
points = gpd.GeoDataFrame(crashes, geometry=gpd.points_from_xy(crashes['longitude'], crashes['latitude']), crs=4326).to_crs(2279)
coords = np.column_stack([points.geometry.x, points.geometry.y])
model = DBSCAN(eps=500, min_samples=2, metric='euclidean')
points['cluster'] = model.fit_predict(coords)
clustered = points[points['cluster'] >= 0].copy()
print('Candidate hotspot areas:', clustered['cluster'].nunique())

hotspots = (clustered.groupby('cluster', as_index=False)
            .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'),
                 serious_injuries=('serious_injuries','sum'), first_year=('year','min'),
                 last_year=('year','max')))
hotspots['affected_bicyclists'] = hotspots['deaths'] + hotspots['serious_injuries']
hotspots = hotspots.sort_values(['affected_bicyclists','deaths','crashes'], ascending=False)
hotspots.to_csv(OUT / 'network_hotspot_areas_2021_present.csv', index=False)
display(hotspots)


In [ ]:
# Give each hotspot a map area and a readable center point.
area_rows = []
for cluster_id, group in clustered.groupby('cluster'):
    area = group.geometry.unary_union.convex_hull.buffer(500)
    area_rows.append({'cluster': cluster_id, 'geometry': area})
hotspot_shapes = gpd.GeoDataFrame(area_rows, crs=2279).merge(hotspots, on='cluster')
hotspot_shapes.to_crs(4326).to_file(OUT / 'network_hotspot_areas_2021_present.geojson', driver='GeoJSON')

fig, ax = plt.subplots(figsize=(9, 8))
hotspot_shapes.plot(ax=ax, column='affected_bicyclists', cmap='OrRd', alpha=.55, legend=True, edgecolor='black')
points.plot(ax=ax, color='steelblue', markersize=12, alpha=.45)
clustered.plot(ax=ax, color='red', markersize=28)
ax.set_title('Candidate bicyclist crash hotspot areas, San Antonio, 2021–2026')
ax.set_axis_off()
fig.tight_layout()
fig.savefig(OUT / 'network_hotspot_areas_2021_present.png', dpi=200)
plt.show()


In [ ]:
# Match each crash to the nearest named street for reporting.
streets_dir = RAW / 'streets'
if not streets_dir.exists():
    streets_dir.mkdir()
    with zipfile.ZipFile(RAW / 'Streets.zip') as z:
        z.extractall(streets_dir)
street_file = next(streets_dir.rglob('*.shp'))
roads = gpd.read_file(street_file).to_crs(2279).rename(columns={'MSAG_NAME':'road_label'})
road_cols = ['road_label', 'geometry']
matched = gpd.sjoin_nearest(points, roads[road_cols], how='left', distance_col='match_distance_ft')
matched = matched.sort_values('match_distance_ft').drop_duplicates('Crash ID')
road_names = (matched.groupby('cluster', as_index=False)
              .agg(roads=('road_label', lambda x: ', '.join(pd.Series(x.dropna().astype(str).str.strip()).drop_duplicates().head(5))),
                   max_match_distance_ft=('match_distance_ft','max')))
report = hotspots.merge(road_names, on='cluster', how='left')
report.to_csv(OUT / 'network_hotspot_areas_2021_present_with_roads.csv', index=False)
display(report)


## Compare with the City's 14 original areas

The City layer is a set of named road-area lines from 2020. The comparison below uses a 150-foot buffer around those lines and counts 2021-present CRIS crashes that fall inside. This shows which original areas have had qualifying crashes since the City created the layer.


In [ ]:
areas_dir = RAW / 'severe_bicyclist_areas'
if not areas_dir.exists():
    areas_dir.mkdir()
    with zipfile.ZipFile(RAW / 'pwSevereBicyclistInjuryAreas.zip') as z:
        z.extractall(areas_dir)
area_file = next(areas_dir.rglob('*.shp'))
areas = gpd.read_file(area_file).to_crs(2279).reset_index(drop=True)
areas['area_id'] = areas.index + 1
areas['area_label'] = areas['FromStreet'].fillna('').astype(str).str.strip() + ' to ' + areas['ToStreet'].fillna('').astype(str).str.strip()
areas['original_2020_total'] = pd.to_numeric(areas['Total_Inju'], errors='coerce')
buffers = areas[['area_id','area_label','original_2020_total','geometry']].copy()
buffers['geometry'] = buffers.geometry.buffer(150)
matches = gpd.sjoin(points, buffers, how='inner', predicate='within')
matches = matches.drop_duplicates(['Crash ID','area_id'])
status = (matches.groupby(['area_id','area_label'], as_index=False)
          .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'),
               serious_injuries=('serious_injuries','sum'), first_year=('year','min'), last_year=('year','max')))
status['affected_bicyclists'] = status['deaths'] + status['serious_injuries']
status = areas[['area_id','area_label','original_2020_total']].merge(status, on=['area_id','area_label'], how='left')
for col in ['crashes','deaths','serious_injuries','affected_bicyclists']:
    status[col] = status[col].fillna(0).astype(int)
status = status.sort_values(['affected_bicyclists','deaths'], ascending=False)
status.to_csv(OUT / 'city_14_area_status_2021_present.csv', index=False)
display(status)
